# Introduction

### Evaluating the Faithfulness of LLM-Generated Explanations for Fact Verification

**Author:** Ilnaza Saifutdinova
**Programme:** B.Sc. Software Engineering — University of Applied Sciences Europe
**Dataset:** FEVER Shared Task Dev Set (19,998 labelled claims)

### Overview
This notebook implements the experimental pipeline for evaluating whether
LLM-generated explanations for fact verification are faithful to the gold
evidence provided in the FEVER benchmark.

**Three prompting conditions:**
- Condition 1: Answer only (baseline)
- Condition 2: Answer + free explanation
- Condition 3: Gold evidence + answer + explanation (RAG)

## Setup and Imports

In [2]:
import json
import pathlib
import time

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from rouge_score import rouge_scorer
import seaborn as sns
import plotly.express as px

# Set seaborn style globally — applies to all matplotlib plots too
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.2)

## 1. Load FEVER Dataset

In [2]:
DATA_PATH = pathlib.Path("data/shared_task_dev.jsonl")

df = pd.read_json(DATA_PATH, lines=True)

print("Dataset shape:", df.shape)
print("\nColumn names:", df.columns.tolist())
df.head()

Dataset shape: (19998, 5)

Column names: ['id', 'verifiable', 'label', 'claim', 'evidence']


,id,verifiable,label,claim,evidence
0,91198,NOT VERIFIABLE,NOT ENOUGH INFO,Colin Kaepernick became a starting quarterback...,"[[[108548, None, None, None]]]"
1,194462,NOT VERIFIABLE,NOT ENOUGH INFO,Tilda Swinton is a vegan.,"[[[227768, None, None, None]]]"
2,137334,VERIFIABLE,SUPPORTS,Fox 2000 Pictures released the film Soul Food.,"[[[289914, 283015, Soul_Food_-LRB-film-RRB-, 0..."
3,166626,NOT VERIFIABLE,NOT ENOUGH INFO,Anne Rice was born in New Jersey.,"[[[191656, None, None, None], [191657, None, N..."
4,111897,VERIFIABLE,REFUTES,Telemundo is a English-language television net...,"[[[131371, 146144, Telemundo, 0]], [[131371, 1..."


## 2. Data Preprocessing

In [3]:
# Label distribution
print("Label distribution:")
print(df["label"].value_counts())
print()

# What does evidence look like for one claim?
print("Evidence structure (SUPPORTS example):")
supports_ex = df[df["label"] == "SUPPORTS"].iloc[0]
print(f"Claim: {supports_ex['claim']}")
print(f"Evidence: {supports_ex['evidence']}")

Label distribution:
label
NOT ENOUGH INFO    6666
SUPPORTS           6666
REFUTES            6666
Name: count, dtype: int64

Evidence structure (SUPPORTS example):
Claim: Fox 2000 Pictures released the film Soul Food.
Evidence: [[[289914, 283015, 'Soul_Food_-LRB-film-RRB-', 0]], [[291259, 284217, 'Soul_Food_-LRB-film-RRB-', 0]], [[293412, 285960, 'Soul_Food_-LRB-film-RRB-', 0]], [[337212, 322620, 'Soul_Food_-LRB-film-RRB-', 0]], [[337214, 322622, 'Soul_Food_-LRB-film-RRB-', 0]]]


### 2.1 Parse Evidence Structure
Flatten the nested evidence pointers and extract readable Wikipedia page names and sentence counts per claim.

In [4]:
def parse_evidence(evidence):
    """Extract unique (page, sentence_id) pairs from evidence sets."""
    seen = set()
    for ev_set in evidence:
        for ev in ev_set:
            if ev[2] is not None:  # ev[2] = wikipedia page
                seen.add((ev[2], ev[3]))  # (page, sentence_id)
    return list(seen)

def clean_page_name(page):
    """Convert Wikipedia URL format to readable name."""
    return page.replace("_", " ").replace("-LRB-", "(").replace("-RRB-", ")")

df["evidence_parsed"] = df["evidence"].apply(parse_evidence)
df["evidence_pages"] = df["evidence_parsed"].apply(
    lambda evs: [clean_page_name(e[0]) for e in evs]
)
df["n_evidence"] = df["evidence_parsed"].apply(len)

print(df[["claim", "label", "evidence_pages", "n_evidence"]].head(5))

                                               claim            label  \
0  Colin Kaepernick became a starting quarterback...  NOT ENOUGH INFO   
1                          Tilda Swinton is a vegan.  NOT ENOUGH INFO   
2     Fox 2000 Pictures released the film Soul Food.         SUPPORTS   
3                  Anne Rice was born in New Jersey.  NOT ENOUGH INFO   
4  Telemundo is a English-language television net...          REFUTES   

                                      evidence_pages  n_evidence  
0                                                 []           0  
1                                                 []           0  
2                                 [Soul Food (film)]           1  
3                                                 []           0  
4  [Telemundo, Hispanic and Latino Americans, Tel...           5  


### 2.2 Normalize Labels
Standardize label formatting and verify class distribution.

In [5]:
# Normalize label formatting (FEVER uses caps, keep consistent)
df["label"] = df["label"].str.strip().str.upper()

# Verify
print("Normalized label counts:")
print(df["label"].value_counts())
print(f"\nAny nulls: {df.isnull().sum().to_dict()}")
print(f"\nDataset shape after preprocessing: {df.shape}")

Normalized label counts:
label
NOT ENOUGH INFO    6666
SUPPORTS           6666
REFUTES            6666
Name: count, dtype: int64

Any nulls: {'id': 0, 'verifiable': 0, 'label': 0, 'claim': 0, 'evidence': 0, 'evidence_parsed': 0, 'evidence_pages': 0, 'n_evidence': 0}

Dataset shape after preprocessing: (19998, 8)


### 2.3 Filter for Stratified Sampling
For the experiments we use claims where evidence is clearly defined:
- SUPPORTS and REFUTES: exactly 1 evidence sentence (cleanest for faithfulness evaluation)
- NOT ENOUGH INFO: no evidence by definition

From 19,998 total claims:
- 4,868 SUPPORTS with single evidence sentence
- 4,898 REFUTES with single evidence sentence
- 6,666 NOT ENOUGH INFO
- 100 sampled per class → 300 claims total (SEED = 2026)

data/
├── shared_task_dev.jsonl    ← full dataset (19,998 claims)
└── sample_300.jsonl         ← your experiment sample (300 claims)

In [6]:
SEED = 2026
N_PER_CLASS = 100

# SUPPORTS and REFUTES: only single-sentence evidence (cleanest for faithfulness eval)
df_sup = df[(df["label"] == "SUPPORTS") & (df["n_evidence"] == 1)]
df_ref = df[(df["label"] == "REFUTES")  & (df["n_evidence"] == 1)]
df_nei = df[df["label"] == "NOT ENOUGH INFO"]

print(f"Available SUPPORTS (1 evidence): {len(df_sup)}")
print(f"Available REFUTES  (1 evidence): {len(df_ref)}")
print(f"Available NEI:                   {len(df_nei)}")

# Stratified sample
sample = pd.concat([
    df_sup.sample(n=N_PER_CLASS, random_state=SEED),
    df_ref.sample(n=N_PER_CLASS, random_state=SEED),
    df_nei.sample(n=N_PER_CLASS, random_state=SEED),
]).reset_index(drop=True)

print(f"\nFinal sample: {len(sample)} claims")
print(sample["label"].value_counts())

sample.to_json("data/sample_300.jsonl", orient="records", lines=True)
print("\nSaved to data/sample_300.jsonl")

Available SUPPORTS (1 evidence): 4868
Available REFUTES  (1 evidence): 4898
Available NEI:                   6666

Final sample: 300 claims
label
SUPPORTS           100
REFUTES            100
NOT ENOUGH INFO    100
Name: count, dtype: int64

Saved to data/sample_300.jsonl


> **Note:** 73% of SUPPORTS (4,868/6,666) and 74% of REFUTES (4,898/6,666)
> claims have single-sentence evidence, confirming the filtered sample
> remains representative of the full dataset.

> **Note:** Also those numbers tell you something useful — roughly 73% of SUPPORTS and 74% of REFUTES claims
> have single-sentence evidence. So filtering to single-sentence didn't throw away much, which means your
> sample is still representative of the dataset overall. Worth one sentence in your methodology chapter.

## 3. Exploratory Data Analysis

Before running any LLM experiments, we visualise the structure of the
sampled dataset to confirm balance and understand claim/evidence characteristics.

In [7]:
pathlib.Path("figures").mkdir(exist_ok=True)

### 3.1 Label Distribution

In [8]:
label_counts = sample["label"].value_counts().reset_index()
label_counts.columns = ["label", "count"]

fig = px.bar(
    label_counts,
    x="label",
    y="count",
    color="label",
    color_discrete_map={
        "SUPPORTS": "#2ecc71",
        "REFUTES": "#e74c3c",
        "NOT ENOUGH INFO": "#95a5a6"
    },
    title="Sample label distribution (n=300)",
    labels={"label": "Label", "count": "Number of claims"},
    text_auto=True
)
fig.update_layout(showlegend=False, plot_bgcolor="white")
fig.show()

### 3.2 Claim Length Distribution
How many words are in each claim, broken down by label.

In [9]:
sample["claim_length"] = sample["claim"].str.split().str.len()

fig = px.histogram(
    sample,
    x="claim_length",
    color="label",
    color_discrete_map={
        "SUPPORTS": "#2ecc71",
        "REFUTES": "#e74c3c",
        "NOT ENOUGH INFO": "#95a5a6"
    },
    barmode="overlay",
    opacity=0.7,
    nbins=30,
    title="Claim length distribution by label",
    labels={"claim_length": "Number of words", "count": "Number of claims"}
)
fig.update_layout(plot_bgcolor="white", legend_title="Label")
fig.show()

>This overlapping histogram will show whether REFUTES claims tend to be longer or shorter than SUPPORTS —
> which is actually an interesting finding for your thesis if there's a difference

> **Note:** All three label classes show similar claim length distributions
> (peak 7–9 words, right-skewed), suggesting claim length is not a
> confounding factor in the experiments.

### 3.3 Evidence Count Distribution
Number of evidence sentences per claim (SUPPORTS and REFUTES only —
NEI has no evidence by definition).

In [10]:
df_ev = sample[sample["label"].isin(["SUPPORTS", "REFUTES"])].copy()

fig = px.histogram(
    df_ev,
    x="n_evidence",
    color="label",
    color_discrete_map={
        "SUPPORTS": "#2ecc71",
        "REFUTES": "#e74c3c",
    },
    barmode="group",
    title="Evidence sentence count (SUPPORTS vs REFUTES)",
    labels={"n_evidence": "Number of evidence sentences", "count": "Number of claims"},
    nbins=10
)
fig.update_layout(plot_bgcolor="white", legend_title="Label")
fig.show()

> **Note:** All sampled SUPPORTS and REFUTES claims have exactly 1 gold
> evidence sentence, ensuring a clean and consistent input for the
> RAG prompting condition (Condition 3).

## 4. LLM Experiments

### 4.1 Setup

In [6]:
import os
import time
import csv
import pathlib
from ollama import Client

# ─── Configuration ───────────────────────────────────────
MODEL_PROVIDER = "ollama"
MODEL_NAME = "deepseek-r1:7b"

# ─── Ollama Client ───────────────────────────────────────
client = Client(host="http://localhost:11434")

try:
    models = client.list()
    print("Ollama connected ✓")
    print("Available models:")
    for m in models.models:
        print(f"  - {m.model}")
except Exception as e:
    print(f"Ollama not running: {e}")

# ─── Load Sample ─────────────────────────────────────────
sample = pd.read_json("data/sample_300.jsonl", lines=True)
print(f"\nSample loaded: {len(sample)} claims")
print(sample["label"].value_counts())

Ollama connected ✓
Available models:
  - deepseek-r1:7b
  - falcon3:7b
  - gemma:latest
  - qwen2:latest
  - mistral:latest
  - llama3:latest

Sample loaded: 300 claims
label
SUPPORTS           100
REFUTES            100
NOT ENOUGH INFO    100
Name: count, dtype: int64


### 4.2 Prompting Conditions

In [7]:
def prompt_condition1(claim: str) -> str:
    """Condition 1: Answer only."""
    return (
        f"Classify the following claim as SUPPORTS, REFUTES, or NOT ENOUGH INFO.\n"
        f"Respond with exactly one of these three labels and nothing else.\n\n"
        f"Claim: {claim}\n"
        f"Label:"
    )

def prompt_condition2(claim: str) -> str:
    """Condition 2: Answer + free explanation."""
    return (
        f"Classify the following claim as SUPPORTS, REFUTES, or NOT ENOUGH INFO.\n"
        f"Then explain your reasoning in 2-3 sentences.\n\n"
        f"Format your response exactly as:\n"
        f"Label: <SUPPORTS|REFUTES|NOT ENOUGH INFO>\n"
        f"Explanation: <your explanation>\n\n"
        f"Claim: {claim}"
    )

def prompt_condition3(claim: str, evidence: str) -> str:
    """Condition 3: Gold evidence + answer + explanation (RAG)."""
    return (
        f"Using the provided evidence, classify the claim as "
        f"SUPPORTS, REFUTES, or NOT ENOUGH INFO.\n"
        f"Then explain your reasoning grounded in the evidence.\n\n"
        f"Format your response exactly as:\n"
        f"Label: <SUPPORTS|REFUTES|NOT ENOUGH INFO>\n"
        f"Explanation: <your explanation>\n\n"
        f"Claim: {claim}\n"
        f"Evidence: {evidence}"
    )

print("Prompting conditions defined")

Prompting conditions defined


### 4.3 Results Storage

In [8]:
pathlib.Path("results").mkdir(exist_ok=True)

def save_result(row: dict):
    """Append one result row to CSV immediately after each call."""
    file_path = f"results/{MODEL_NAME.replace(':', '_')}_results.csv"
    file_exists = pathlib.Path(file_path).exists()

    with open(file_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=row.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)

print("Results storage ready")

Results storage ready


### 4.4 LLM Call + Response Parser

In [9]:
def call_llm(prompt: str) -> str:
    """Call Ollama model."""
    response = client.chat(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}]
    )
    return response["message"]["content"].strip()

def parse_response(raw: str, condition: int) -> dict:
    """Extract label and explanation from model output."""
    label, explanation = None, None

    if condition == 1:
        raw_clean = raw.strip().upper()
        for l in ["NOT ENOUGH INFO", "SUPPORTS", "REFUTES"]:
            if l in raw_clean:
                label = l
                break
    else:
        for line in raw.splitlines():
            if line.startswith("Label:"):
                label_raw = line.replace("Label:", "").strip().upper()
                for l in ["NOT ENOUGH INFO", "SUPPORTS", "REFUTES"]:
                    if l in label_raw:
                        label = l
                        break
            elif line.startswith("Explanation:"):
                explanation = line.replace("Explanation:", "").strip()

    return {"predicted_label": label, "explanation": explanation}

print("Parser ready")

Parser ready


### 4.5 Run Experiments

In [ ]:
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

CONDITIONS = [1, 2, 3]
total = len(sample) * len(CONDITIONS)
count = 0
errors = 0

print(f"Starting experiments: {len(sample)} claims × {len(CONDITIONS)} conditions")
print(f"Model: {MODEL_NAME}")
print(f"Total calls: {total}\n")

for _, row in sample.iterrows():
    claim = row["claim"]
    gold_label = row["label"]

    # Get evidence for condition 3
    evidence_list = row.get("evidence_pages", [])
    evidence = evidence_list[0] if evidence_list else "No evidence available"

    for condition in CONDITIONS:
        try:
            # Build prompt
            if condition == 1:
                prompt = prompt_condition1(claim)
            elif condition == 2:
                prompt = prompt_condition2(claim)
            else:
                prompt = prompt_condition3(claim, evidence)

            # Call model
            raw = call_llm(prompt)
            parsed = parse_response(raw, condition)

            # Compute ROUGE-L if explanation exists
            rouge_score = None
            if parsed["explanation"] and evidence != "No evidence available":
                score = scorer.score(evidence, parsed["explanation"])
                rouge_score = round(score["rougeL"].fmeasure, 4)

            # Save immediately
            save_result({
                "claim_id": row["id"],
                "claim": claim,
                "gold_label": gold_label,
                "condition": f"condition_{condition}",
                "model": MODEL_NAME,
                "predicted_label": parsed["predicted_label"],
                "explanation": parsed["explanation"],
                "rouge_score": rouge_score,
                "raw_response": raw
            })

            count += 1
            if count % 30 == 0:
                print(f"Progress: {count}/{total} ({round(count/total*100)}%)")

        except Exception as e:
            errors += 1
            print(f"Error on claim {row['id']} condition {condition}: {e}")
            continue

        time.sleep(0.2)

print(f"\nDone! {count} results saved")
print(f"  File: results/{MODEL_NAME.replace(':', '_')}_results.csv")
print(f"  Errors: {errors}")

Starting experiments: 300 claims × 3 conditions
Model: deepseek-r1:7b
Total calls: 900

Progress: 30/900 (3%)
Progress: 60/900 (7%)
Progress: 90/900 (10%)
Progress: 120/900 (13%)
Progress: 150/900 (17%)
Progress: 180/900 (20%)
Progress: 210/900 (23%)
Progress: 240/900 (27%)
Progress: 270/900 (30%)
Progress: 300/900 (33%)
Progress: 330/900 (37%)
Progress: 360/900 (40%)


In [3]:
# Check what's already saved
existing = pd.read_csv("results/deepseek-r1_7b_results.csv")
completed_ids = set(zip(existing["claim_id"], existing["condition"]))
print(f"Already completed: {len(existing)} rows")
print(f"Unique claim+condition pairs: {len(completed_ids)}")

Already completed: 373 rows
Unique claim+condition pairs: 373


In [4]:
print(existing["condition"].value_counts())
print(f"Unique claim IDs: {existing['claim_id'].nunique()}")

condition
condition_1    125
condition_2    124
condition_3    124
Name: count, dtype: int64
Unique claim IDs: 125


#### DeepSeek continue

In [10]:
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

# Load already completed pairs
try:
    existing = pd.read_csv(f"results/{MODEL_NAME.replace(':', '_')}_results.csv")
    completed_ids = set(zip(existing["claim_id"].astype(str), existing["condition"]))
    print(f"Resuming from {len(existing)} existing rows")
except FileNotFoundError:
    completed_ids = set()
    print("Starting fresh")

CONDITIONS = [1, 2, 3]
total = len(sample) * len(CONDITIONS)
count = len(existing) if completed_ids else 0
errors = 0

print(f"Remaining calls: {total - count}")
print(f"Model: {MODEL_NAME}\n")

for _, row in sample.iterrows():
    claim = row["claim"]
    gold_label = row["label"]
    evidence_list = row.get("evidence_pages", [])
    evidence = evidence_list[0] if evidence_list else "No evidence available"

    for condition in CONDITIONS:
        # Skip already completed
        if (str(row["id"]), f"condition_{condition}") in completed_ids:
            continue

        try:
            if condition == 1:
                prompt = prompt_condition1(claim)
            elif condition == 2:
                prompt = prompt_condition2(claim)
            else:
                prompt = prompt_condition3(claim, evidence)

            raw = call_llm(prompt)
            parsed = parse_response(raw, condition)

            rouge_score = None
            if parsed["explanation"] and evidence != "No evidence available":
                score = scorer.score(evidence, parsed["explanation"])
                rouge_score = round(score["rougeL"].fmeasure, 4)

            save_result({
                "claim_id": row["id"],
                "claim": claim,
                "gold_label": gold_label,
                "condition": f"condition_{condition}",
                "model": MODEL_NAME,
                "predicted_label": parsed["predicted_label"],
                "explanation": parsed["explanation"],
                "rouge_score": rouge_score,
                "raw_response": raw
            })

            count += 1
            if count % 30 == 0:
                print(f"Progress: {count}/{total} ({round(count/total*100)}%)")

        except Exception as e:
            errors += 1
            print(f"Error on claim {row['id']} condition {condition}: {e}")
            continue

        time.sleep(0.2)

print(f"\nDone! {count} results saved")
print(f"Errors: {errors}")

Resuming from 373 existing rows
Remaining calls: 527
Model: deepseek-r1:7b

Progress: 390/900 (43%)
Progress: 420/900 (47%)
Progress: 450/900 (50%)
Progress: 480/900 (53%)
Progress: 510/900 (57%)
Progress: 540/900 (60%)
Progress: 570/900 (63%)
Progress: 600/900 (67%)
Progress: 630/900 (70%)
Progress: 660/900 (73%)
Progress: 690/900 (77%)
Progress: 720/900 (80%)
Progress: 750/900 (83%)
Progress: 780/900 (87%)
Progress: 810/900 (90%)
Progress: 840/900 (93%)
Progress: 870/900 (97%)
Progress: 900/900 (100%)

Done! 900 results saved
Errors: 0


### 4.6 Clean Results
Remove duplicate rows caused by any saving interruptions during experiments.

In [19]:
results_df = pd.read_csv("results/llama3_latest_results.csv")
print(f"Rows before cleanup: {len(results_df)}")

# Remove duplicates keeping first occurrence
results_df = results_df.drop_duplicates(
    subset=["claim_id", "condition"],
    keep="first"
)
print(f"Rows after cleanup: {len(results_df)}")

# Save cleaned version
results_df.to_csv("results/llama3_latest_results.csv", index=False)
print("Saved clean results")

# Quick check
print(f"\nCondition counts:")
print(results_df["condition"].value_counts())
print(f"\nLabel counts:")
print(results_df["gold_label"].value_counts())

Rows before cleanup: 907
Rows after cleanup: 900
Saved clean results

Condition counts:
condition
condition_1    300
condition_2    300
condition_3    300
Name: count, dtype: int64

Label counts:
gold_label
SUPPORTS           300
REFUTES            300
NOT ENOUGH INFO    300
Name: count, dtype: int64


### 4.7 Results Overview
Quick accuracy and ROUGE-L check across three prompting conditions.

#### Llama3 Results Overview

In [21]:
from sklearn.metrics import accuracy_score

print(f"=== LLAMA3 RESULTS OVERVIEW ===\n")

for condition in ["condition_1", "condition_2", "condition_3"]:
    subset = results_df[results_df["condition"] == condition].copy()

    # Drop rows where model failed to predict
    valid = subset.dropna(subset=["predicted_label"])
    failed = len(subset) - len(valid)

    # Accuracy
    acc = accuracy_score(valid["gold_label"], valid["predicted_label"])

    print(f"{condition}:")
    print(f"  Accuracy:     {acc:.3f} ({acc*100:.1f}%)")
    print(f"  Failed parses: {failed}")

    # ROUGE-L for conditions 2 and 3
    if condition in ["condition_2", "condition_3"]:
        rouge_mean = valid["rouge_score"].mean()
        print(f"  Avg ROUGE-L:  {rouge_mean:.4f}")
    print()

=== LLAMA3 RESULTS OVERVIEW ===

condition_1:
  Accuracy:     0.543 (54.3%)
  Failed parses: 0

condition_2:
  Accuracy:     0.542 (54.2%)
  Failed parses: 1
  Avg ROUGE-L:  0.1013

condition_3:
  Accuracy:     0.659 (65.9%)
  Failed parses: 1
  Avg ROUGE-L:  0.0847



#### Mistral Results Overview

In [27]:
from sklearn.metrics import accuracy_score

results_df_mistral = pd.read_csv("results/mistral_latest_results.csv")
print(f"=== MISTRAL RESULTS OVERVIEW ===\n")

for condition in ["condition_1", "condition_2", "condition_3"]:
    subset = results_df_mistral[results_df_mistral["condition"] == condition].copy()

    # Drop rows where model failed to predict
    valid = subset.dropna(subset=["predicted_label"])
    failed = len(subset) - len(valid)

    # Accuracy
    acc = accuracy_score(valid["gold_label"], valid["predicted_label"])

    print(f"{condition}:")
    print(f"  Accuracy:     {acc:.3f} ({acc*100:.1f}%)")
    print(f"  Failed parses: {failed}")

    # ROUGE-L for conditions 2 and 3
    if condition in ["condition_2", "condition_3"]:
        rouge_mean = valid["rouge_score"].mean()
        print(f"  Avg ROUGE-L:  {rouge_mean:.4f}")
    print()

=== MISTRAL RESULTS OVERVIEW ===

condition_1:
  Accuracy:     0.490 (49.0%)
  Failed parses: 0

condition_2:
  Accuracy:     0.407 (40.7%)
  Failed parses: 0
  Avg ROUGE-L:  0.0965

condition_3:
  Accuracy:     0.497 (49.7%)
  Failed parses: 0
  Avg ROUGE-L:  0.1069



#### Qwen2 Results Overview

In [33]:
from sklearn.metrics import accuracy_score

results_df_mistral = pd.read_csv("results/qwen2_latest_results.csv")
print(f"=== QWEN2 RESULTS OVERVIEW ===\n")

for condition in ["condition_1", "condition_2", "condition_3"]:
    subset = results_df_mistral[results_df_mistral["condition"] == condition].copy()

    # Drop rows where model failed to predict
    valid = subset.dropna(subset=["predicted_label"])
    failed = len(subset) - len(valid)

    # Accuracy
    acc = accuracy_score(valid["gold_label"], valid["predicted_label"])

    print(f"{condition}:")
    print(f"  Accuracy:     {acc:.3f} ({acc*100:.1f}%)")
    print(f"  Failed parses: {failed}")

    # ROUGE-L for conditions 2 and 3
    if condition in ["condition_2", "condition_3"]:
        rouge_mean = valid["rouge_score"].mean()
        print(f"  Avg ROUGE-L:  {rouge_mean:.4f}")
    print()

=== QWEN2 RESULTS OVERVIEW ===

condition_1:
  Accuracy:     0.549 (54.9%)
  Failed parses: 5

condition_2:
  Accuracy:     0.593 (59.3%)
  Failed parses: 0
  Avg ROUGE-L:  0.0969

condition_3:
  Accuracy:     0.670 (67.0%)
  Failed parses: 0
  Avg ROUGE-L:  0.0791



#### Gemma Results Overview

In [40]:
from sklearn.metrics import accuracy_score

results_df_mistral = pd.read_csv("results/gemma_latest_results.csv")
print(f"=== GEMMA RESULTS OVERVIEW ===\n")

for condition in ["condition_1", "condition_2", "condition_3"]:
    subset = results_df_mistral[results_df_mistral["condition"] == condition].copy()

    # Drop rows where model failed to predict
    valid = subset.dropna(subset=["predicted_label"])
    failed = len(subset) - len(valid)

    # Accuracy
    acc = accuracy_score(valid["gold_label"], valid["predicted_label"])

    print(f"{condition}:")
    print(f"  Accuracy:     {acc:.3f} ({acc*100:.1f}%)")
    print(f"  Failed parses: {failed}")

    # ROUGE-L for conditions 2 and 3
    if condition in ["condition_2", "condition_3"]:
        rouge_mean = valid["rouge_score"].mean()
        print(f"  Avg ROUGE-L:  {rouge_mean:.4f}")
    print()

=== GEMMA RESULTS OVERVIEW ===

condition_1:
  Accuracy:     0.510 (51.0%)
  Failed parses: 0

condition_2:
  Accuracy:     0.522 (52.2%)
  Failed parses: 0
  Avg ROUGE-L:  0.1818

condition_3:
  Accuracy:     0.639 (63.9%)
  Failed parses: 0
  Avg ROUGE-L:  0.1437



#### Falcon3:7b Results Overview

In [9]:
from sklearn.metrics import accuracy_score

results_df_mistral = pd.read_csv("results/falcon3_7b_results.csv")
print(f"=== FALCON RESULTS OVERVIEW ===\n")

for condition in ["condition_1", "condition_2", "condition_3"]:
    subset = results_df_mistral[results_df_mistral["condition"] == condition].copy()

    # Drop rows where model failed to predict
    valid = subset.dropna(subset=["predicted_label"])
    failed = len(subset) - len(valid)

    # Accuracy
    acc = accuracy_score(valid["gold_label"], valid["predicted_label"])

    print(f"{condition}:")
    print(f"  Accuracy:     {acc:.3f} ({acc*100:.1f}%)")
    print(f"  Failed parses: {failed}")

    # ROUGE-L for conditions 2 and 3
    if condition in ["condition_2", "condition_3"]:
        rouge_mean = valid["rouge_score"].mean()
        print(f"  Avg ROUGE-L:  {rouge_mean:.4f}")
    print()

=== FALCON RESULTS OVERVIEW ===

condition_1:
  Accuracy:     0.369 (36.9%)
  Failed parses: 5

condition_2:
  Accuracy:     0.411 (41.1%)
  Failed parses: 3
  Avg ROUGE-L:  0.1264

condition_3:
  Accuracy:     0.537 (53.7%)
  Failed parses: 4
  Avg ROUGE-L:  0.1002



#### DeepSeek Results Overview

In [11]:
from sklearn.metrics import accuracy_score

results_df_mistral = pd.read_csv("results/deepseek-r1_7b_results.csv")
print(f"=== DEEPSEEK-R1 RESULTS OVERVIEW ===\n")

for condition in ["condition_1", "condition_2", "condition_3"]:
    subset = results_df_mistral[results_df_mistral["condition"] == condition].copy()

    # Drop rows where model failed to predict
    valid = subset.dropna(subset=["predicted_label"])
    failed = len(subset) - len(valid)

    # Accuracy
    acc = accuracy_score(valid["gold_label"], valid["predicted_label"])

    print(f"{condition}:")
    print(f"  Accuracy:     {acc:.3f} ({acc*100:.1f}%)")
    print(f"  Failed parses: {failed}")

    # ROUGE-L for conditions 2 and 3
    if condition in ["condition_2", "condition_3"]:
        rouge_mean = valid["rouge_score"].mean()
        print(f"  Avg ROUGE-L:  {rouge_mean:.4f}")
    print()

=== DEEPSEEK-R1 RESULTS OVERVIEW ===

condition_1:
  Accuracy:     0.438 (43.8%)
  Failed parses: 12

condition_2:
  Accuracy:     0.425 (42.5%)
  Failed parses: 1
  Avg ROUGE-L:  0.1155

condition_3:
  Accuracy:     0.393 (39.3%)
  Failed parses: 0
  Avg ROUGE-L:  0.1297

